# 21. 데이터형 확장 후보 (imine_2, het-C-het, quaternary_nitrogen 등)

## 이번 노트북에서 할 것
- imine_2가 imine_1(oxime/general)과 화학적으로 다른 하위형인지 실제 매치로 확인
- het-C-het_not_in_ring 실제 구조 확인 (이름이 일반적이라 좁혀야 함)
- quaternary_nitrogen_1/2 검토 (4급 질소, 국소 치환 개념과 안 맞을 가능성 검토)
- 확인 결과에 따라 replacement_library.py 반영 여부 결정
- 문헌형(beta-keto/anhydride, quinone_A 등)은 학생이 논문 확인 후 별도 진행

## 간략한 정리 (20까지)
- 라이브러리 13개 규칙 확정 (aniline은 아세트아미드/BCP 두 candidate로 통합)
- 중요 발견: 같은 부위의 "변화 폭이 다른 대안 해법"을 별도 규칙으로 나누면
  LLM 판단이 보수적 옵션으로 편향(7/8, 87.5%); 하나의 규칙+복수 candidate로
  통합하면 균형잡힌 분자별 맥락 판단(5:3) 발생 - 향후 규칙 설계 원칙으로 확정
- Git author 이메일 오류 3건 rebase로 소급 수정, 잔디 정상화
- 6가지 편집 타입 완비: fragment-cut, replace_element, add_substituent,
  reduce_bond, replace_ring, replace_multi
- test set은 여전히 미사용, valid set(seed=7)으로만 개발/검증

## 다음에 해야 할 것 (오늘 끝나면)
- 최종 valid set 재검증 (전체 규칙 반영, 커버리지/성공률/3-endpoint)
- 학생 승인 시 test set 최종 1회 검증
- 제안서 반영, 문헌형 확장은 학생 진행에 따라 병행

In [1]:
# 셀 1
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 44.4 MB/s eta 0:00:00


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 239 (delta 125), reused 166 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (239/239), 619.86 KiB | 2.26 MiB/s, done.
Resolving deltas: 100% (125/125), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib
from rdkit import Chem
from rdkit.Chem import rdMMPA

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.agent

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use
from src.tools.atom_editor import apply_atom_edit_from_rule

data = load_tox21_clean(random_state=7)
print("도구 로드 확인 완료 (valid set 사용, test set 보류)")

[07:35:02] WARNING: not removing hydrogen atom without neighbors
[07:35:03] Explicit valence for atom # 8 Al, 6, is greater than permitted
[07:35:03] Explicit valence for atom # 3 Al, 6, is greater than permitted
[07:35:03] Explicit valence for atom # 4 Al, 6, is greater than permitted
[07:35:03] Explicit valence for atom # 4 Al, 6, is greater than permitted
[07:35:03] Explicit valence for atom # 9 Al, 6, is greater than permitted
[07:35:03] Explicit valence for atom # 5 Al, 6, is greater than permitted
[07:35:04] Explicit valence for atom # 16 Al, 6, is greater than permitted
[07:35:04] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[07:35:04] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료 (valid set 사용, test set 보류)


In [5]:
target_names_v21 = ["imine_2", "het-C-het_not_in_ring", "quaternary_nitrogen_1", "quaternary_nitrogen_2"]
examples_v21 = {}

for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] in target_names_v21 and p['rule_name'] not in examples_v21:
            examples_v21[p['rule_name']] = (s, p['atom_indices'])
    if len(examples_v21) == len(target_names_v21):
        break

for name, (smi, indices) in examples_v21.items():
    print(f"\n{name}: {smi}")
    mol = Chem.MolFromSmiles(smi)
    for idx in indices:
        atom = mol.GetAtomWithIdx(idx)
        print(f"  인덱스 {idx}: {atom.GetSymbol()} (방향족: {atom.GetIsAromatic()}, 전하: {atom.GetFormalCharge()}, 이웃: {[n.GetSymbol() for n in atom.GetNeighbors()]})")

print(f"\n확보된 규칙: {list(examples_v21.keys())}")


het-C-het_not_in_ring: CCCOC(OCCC)OCCC
  인덱스 3: O (방향족: False, 전하: 0, 이웃: ['C', 'C'])
  인덱스 4: C (방향족: False, 전하: 0, 이웃: ['O', 'O', 'O'])
  인덱스 5: O (방향족: False, 전하: 0, 이웃: ['C', 'C'])

imine_2: CCOC(=O)c1ccc(OC(=O)CCCCCNC(=N)N)cc1
  인덱스 17: N (방향족: False, 전하: 0, 이웃: ['C', 'C'])
  인덱스 18: C (방향족: False, 전하: 0, 이웃: ['N', 'N', 'N'])
  인덱스 19: N (방향족: False, 전하: 0, 이웃: ['C'])

quaternary_nitrogen_2: CC[N+](CC)(CC)CCOc1ccc(/C=C/c2ccccc2)cc1
  인덱스 1: C (방향족: False, 전하: 0, 이웃: ['C', 'N'])
  인덱스 2: N (방향족: False, 전하: 1, 이웃: ['C', 'C', 'C', 'C'])
  인덱스 3: C (방향족: False, 전하: 0, 이웃: ['N', 'C'])
  인덱스 5: C (방향족: False, 전하: 0, 이웃: ['N', 'C'])
  인덱스 7: C (방향족: False, 전하: 0, 이웃: ['N', 'C'])

quaternary_nitrogen_1: CCCCCC[n+]1ccccc1.F[B-](F)(F)F
  인덱스 5: C (방향족: False, 전하: 0, 이웃: ['C', 'N'])
  인덱스 6: N (방향족: True, 전하: 1, 이웃: ['C', 'C', 'C'])
  인덱스 7: C (방향족: True, 전하: 0, 이웃: ['N', 'C'])
  인덱스 11: C (방향족: True, 전하: 0, 이웃: ['C', 'N'])

확보된 규칙: ['het-C-het_not_in_ring', 'imine_2', 'quaternary_nitrogen_

In [6]:
test_ortho = "CCCOC(OCCC)OCCC"
pattern_ortho = Chem.MolFromSmarts("C(OC)(OC)OC")
print("패턴 크기:", pattern_ortho.GetNumAtoms())
print("매치:", Chem.MolFromSmiles(test_ortho).HasSubstructMatch(pattern_ortho))

패턴 크기: 7
매치: True


In [7]:
pattern_ortho2 = Chem.MolFromSmarts("[CX4](O)(O)O")
print("패턴 크기:", pattern_ortho2.GetNumAtoms())
mol_ortho = Chem.MolFromSmiles(test_ortho)
print("매치:", mol_ortho.HasSubstructMatch(pattern_ortho2))

패턴 크기: 4
매치: True


In [8]:
!cat src/tools/atom_editor.py

from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행.
    candidate마다 다른 edit_type을 가질 수 있음 (예: 같은 문제에 대해
    작은 변화(치환기 하나 추가)와 큰 변화(고리 전체 교체)를 후보로 병렬 제시)."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
  

In [9]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행.
    candidate마다 다른 edit_type을 가질 수 있음 (예: 같은 문제에 대해
    작은 변화(치환기 하나 추가)와 큰 변화(고리 전체 교체)를 후보로 병렬 제시)."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        # remove_idx_in_pattern: 제거할 치환기 시작 원자(패턴 내 위치)
        # upgrade_bond_to_idx_in_pattern: 남아서 이중결합으로 승격될 원자(패턴 내 위치)
        # center_idx_in_pattern: 중심 원자(패턴 내 위치) - 이 원자 쪽으로는 삭제가 번지지 않도록 차단
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        # remove_idx에서 시작해, center_idx 쪽으로는 넘어가지 않으면서
        # 연결된 원자들을 전부 찾아 삭제 대상으로 표시 (치환기 전체 제거)
        to_remove = set()
        stack = [remove_idx]
        visited = {center_idx}
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        for atom in check_mol.GetAtoms():
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [10]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "target_idx_in_pattern": 0,
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "add_substituent", "param": "C(=O)C",
             "target_idx_in_pattern": 0,
             "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
             "anchor_indices_in_pattern": (0, 5),
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 "
                          "변화 폭이 크지만, 물성 개선 효과도 더 큼"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 "
                          "메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. "
                          "오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 "
                          "낮춤 (학생 확인 예정: ScienceDirect catechol overview, "
                          "PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
    "het-C-het_not_in_ring": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4](O)(O)O",
        "candidates": [
            {"edit_type": "remove_substituent",
             "center_idx_in_pattern": 0,
             "remove_idx_in_pattern": 1,
             "upgrade_bond_to_idx_in_pattern": 2,
             "name": "ester (one alkoxy removed, C=O formed)",
             "rationale": "오르토에스터(탄소 하나에 알콕시기 3개)는 가수분해에 매우 "
                          "민감하여 알데히드/에스터로 쉽게 분해되며 대사 불안정성을 "
                          "일으킴. 알콕시기 하나를 제거하고 남은 산소를 카르보닐로 "
                          "승격시켜 일반적인 에스터로 전환, 가수분해 반응성을 낮춤 "
                          "(검증 필요)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [11]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

test_ortho = "CCCOC(OCCC)OCCC"
result_ortho = propose_fix(test_ortho, "het-C-het_not_in_ring", candidate_idx=0)
print(result_ortho)

# 회귀 테스트
print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("CC(C)OC(=S)[S-]", "thiol_1", candidate_idx=0))

None

=== 회귀 테스트 ===
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'CC(C)OC(N)=O', 'candidate_used': 'carbamate (O,N replacing S,S)', 'rationale': '디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 독성 기전)을 제거함 (검증 필요)', 'is_valid': True}


[07:47:37] Explicit valence for atom # 1 O, 3, is greater than permitted


In [12]:
mol_debug = Chem.MolFromSmiles(test_ortho)
pattern_debug = Chem.MolFromSmarts("[CX4](O)(O)O")
match_debug = mol_debug.GetSubstructMatches(pattern_debug)[0]
print("매치:", match_debug)
for idx in match_debug:
    atom = mol_debug.GetAtomWithIdx(idx)
    print(f"  idx={idx}: {atom.GetSymbol()}, 이웃={[n.GetIdx() for n in atom.GetNeighbors()]}")

매치: (4, 3, 5, 9)
  idx=4: C, 이웃=[3, 5, 9]
  idx=3: O, 이웃=[2, 4]
  idx=5: O, 이웃=[4, 6]
  idx=9: O, 이웃=[4, 10]


In [13]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행.
    candidate마다 다른 edit_type을 가질 수 있음 (예: 같은 문제에 대해
    작은 변화(치환기 하나 추가)와 큰 변화(고리 전체 교체)를 후보로 병렬 제시)."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        # remove_idx_in_pattern: 완전히 제거할 치환기 시작 원자(패턴 내 위치)
        # upgrade_bond_to_idx_in_pattern: 남아서 이중결합으로 승격될 원자(패턴 내 위치).
        #   이 원자에 붙은 알킬기(중심원자 방향 제외)도 함께 제거해야 카르보닐로 완성됨
        # center_idx_in_pattern: 중심 원자(패턴 내 위치)
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}

        # 1) remove_idx 쪽 치환기 전체 삭제 대상 수집
        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        # 2) upgrade_idx는 남기되, 거기 붙은 알킬기(중심/이미 삭제대상 제외)도 제거
        visited.add(upgrade_idx)
        upgrade_atom = mol.GetAtomWithIdx(upgrade_idx)
        for n in upgrade_atom.GetNeighbors():
            if n.GetIdx() != center_idx and n.GetIdx() not in to_remove:
                stack2 = [n.GetIdx()]
                while stack2:
                    cur2 = stack2.pop()
                    if cur2 in visited:
                        continue
                    visited.add(cur2)
                    to_remove.add(cur2)
                    for n2 in mol.GetAtomWithIdx(cur2).GetNeighbors():
                        if n2.GetIdx() not in visited:
                            stack2.append(n2.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        for atom in check_mol.GetAtoms():
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [14]:
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

result_ortho2 = propose_fix("CCCOC(OCCC)OCCC", "het-C-het_not_in_ring", candidate_idx=0)
print(result_ortho2)

print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("CC(C)OC(=S)[S-]", "thiol_1", candidate_idx=0))

{'new_smiles': 'CCCOC=O', 'candidate_used': 'ester (one alkoxy removed, C=O formed)', 'rationale': '오르토에스터(탄소 하나에 알콕시기 3개)는 가수분해에 매우 민감하여 알데히드/에스터로 쉽게 분해되며 대사 불안정성을 일으킴. 알콕시기 하나를 제거하고 남은 산소를 카르보닐로 승격시켜 일반적인 에스터로 전환, 가수분해 반응성을 낮춤 (검증 필요)', 'is_valid': True}

=== 회귀 테스트 ===
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'CC(C)OC(N)=O', 'candidate_used': 'carbamate (O,N replacing S,S)', 'rationale': '디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 독성 기전)을 제거함 (검증 필요)', 'is_valid': True}


In [15]:
all_rules_v21 = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
print("현재 라이브러리:", all_rules_v21, f"({len(all_rules_v21)}개)\n")

test_examples_v21 = {}
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] in all_rules_v21 and p['rule_name'] not in test_examples_v21:
            test_examples_v21[p['rule_name']] = s
    if len(test_examples_v21) == len(all_rules_v21):
        break

print(f"예시 확보: {len(test_examples_v21)}/{len(all_rules_v21)}개")
missing_v21 = set(all_rules_v21) - set(test_examples_v21.keys())
if missing_v21:
    print("못 찾은 규칙:", missing_v21)

print("\n=== 규칙별 전체 candidate 테스트 ===")
all_ok = True
for rule, smi in test_examples_v21.items():
    info = get_replacement_candidates(rule)
    for idx in range(len(info['candidates'])):
        result = propose_fix(smi, rule, candidate_idx=idx)
        ok = result is not None and result.get('is_valid', False)
        if not ok:
            all_ok = False
        status = "✅" if ok else "❌"
        print(f"  {rule} [candidate {idx}]: {status}")
        if not ok:
            print(f"    -> 분자: {smi}, 결과: {result}")

print(f"\n전체 결과: {'모두 통과 ✅' if all_ok else '일부 실패 ❌ - 위 로그 확인 필요'}")

현재 라이브러리: ['nitro_group', 'aldehyde', 'Michael_acceptor_1', 'acid_halide', 'alkyl_halide', 'aniline', 'Sulfonic_acid_2', 'imine_1_oxime', 'imine_1_general', 'catechol', 'Thiocarbonyl_group', 'thiol_2', 'thiol_1', 'het-C-het_not_in_ring'] (14개)

예시 확보: 14/14개

=== 규칙별 전체 candidate 테스트 ===
  aniline [candidate 0]: ✅
  aniline [candidate 1]: ✅
  het-C-het_not_in_ring [candidate 0]: ✅
  nitro_group [candidate 0]: ✅
  nitro_group [candidate 1]: ✅
  nitro_group [candidate 2]: ✅
  imine_1_general [candidate 0]: ✅
  aldehyde [candidate 0]: ✅
  aldehyde [candidate 1]: ✅
  alkyl_halide [candidate 0]: ✅
  alkyl_halide [candidate 1]: ✅
  catechol [candidate 0]: ✅
  Michael_acceptor_1 [candidate 0]: ✅
  Thiocarbonyl_group [candidate 0]: ✅
  acid_halide [candidate 0]: ✅
  acid_halide [candidate 1]: ✅
  Sulfonic_acid_2 [candidate 0]: ✅
  Sulfonic_acid_2 [candidate 1]: ✅
  imine_1_oxime [candidate 0]: ✅
  thiol_2 [candidate 0]: ✅
  thiol_2 [candidate 1]: ✅
  thiol_1 [candidate 0]: ✅

전체 결과: 모두 통과 ✅


In [16]:
!git add -A
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/atom_editor.py
	modified:   src/tools/replacement_library.py



In [17]:
!git commit -m "Full regression verification: all 14 rules, all candidates pass automated pipeline test (detect_toxicophores + propose_fix) after remove_substituent addition and multiple file rewrites"
!git push origin main

[main 84f85a5] Full regression verification: all 14 rules, all candidates pass automated pipeline test (detect_toxicophores + propose_fix) after remove_substituent addition and multiple file rewrites
 2 files changed, 73 insertions(+)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.80 KiB | 1.80 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   0d71890..84f85a5  main -> main
